In [1]:
# the way I generated the PARM predictions/the way they output files each promoter/ism is output in a chrom:start-stop directory
# each directory contains:
#
# 1. hits_chrom:start-end.txt.gz
# 2. mutagenesis_chrrom:start-end.txt.gz
#
# 1 contains the hocomoco output from PARM's ISM calling, they consider hits as those with an abs(rho) > 0.75
# for this data I want to convert this to be usable with tangermeme plotting structure 
# we'll return a dictionary of [promoter/enhancer][filtered_hits] - we will reformat the DF to the shape:
#                       example_idx 	start 	end 	attribution 	rho 	enhancer_id 	
#
# matching columns are as follows:
# 'name_motif' -> example_idx
# 'start' -> 'start
# 'end' -> end
# 'att' -> attribution
# 'rho' -> rho (shouldn't be needed for plotting but good for sanity checking)
# 'enhancer_id' -> shouldn't really be necessary again as we won't be calling everything together but may be useful
#
# 2 contains the actual predictions from PARM
# we'll save those as 'raw' tensors in a dictionary with enhancer key, tesor value pairs
# we'll also save those as contribution scores for plotting comparisons
# will also be useful to save in seqlet calling compatible tensor to make comparisons between our methods and theirs   

In [2]:
# import functions
import glob
import torch
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import seaborn as sns
from collections import Counter
import pickle

In [3]:
# define function for generating an interval : annotation dictionary from a input bed file
# this will be used in both filtering functions to 
def interval_dict (path2bed):
    # open bed file as dataframe
    bed_df = pd.read_csv(path2bed, sep = '\t', header = None)
    # convert the chrom, start, end into key
    bed_key = [f'{chrom}:{str(start)}-{str(end)}' for chrom, start, end in zip(bed_df[0], bed_df[1], bed_df[2])]
    # get ids for each interval
    gene_vals = list(bed_df[3])
    # make dictionary and return
    return dict(zip(bed_key, gene_vals))


In [4]:
ann_dict = interval_dict('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/processed_data/GRCh38-dELS-only.bed')

In [5]:
def enhancer_dict (path2bed):
    # open bed file as dataframe
    bed_df = pd.read_csv(path2bed, sep = '\t', header = None)
    # convert the chrom, start, end into key
    bed_vals = [f'{chrom}:{str(start)}-{str(end)}' for chrom, start, end in zip(bed_df[0], bed_df[1], bed_df[2])]
    # get ids for each interval
    gene_keys = list(bed_df[3])
    # make dictionary and return
    return dict(zip(gene_keys, bed_vals))

In [6]:
enh_dict = enhancer_dict('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/processed_data/GRCh38-dELS-only.bed')

In [78]:
# save ann and enh dicts to disk
annPath = '/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/scripts/parm_tf_processing/dELS_ann_dict.pkl'
enhPath = '/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/scripts/parm_tf_processing/dELS_enh_dict.pkl'

with open(annPath, "wb") as file:
    pickle.dump(ann_dict, file)

with open(enhPath, "wb") as file:
    pickle.dump(enh_dict, file)

In [14]:
# get all k562 predictions
k562_folders = glob.glob('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/parm_preds/final_sat_mut_preds/k562/*')
# get all hepg2 predictions
hepg2_folders = glob.glob('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/parm_preds/final_sat_mut_preds/hepg2/*')

In [44]:
### let's define the tensor output function ###
### PARM uses REF - ALT for skew calculations, no need to multiply (i think... (092625)) ###
def parm2tangermeme_tensors (path2folders, pos_id_dict):
    # define dictionaries
    raw_tensor_dict = {}
    seqlet_tensor_dict = {}
    plotLogo_tensor_dict = {}
    oneHot_tensor_dict = {}
    for folder in tqdm(path2folders):
        # get all folders in this chunk
        chunk_folders = os.listdir(folder)
        # iterate through the folders in this chunk
        for chunk in chunk_folders:
            # get current wd and store as variable for returning to root of folders
            # pull the enhancer/gene name
            element = chunk.split('/')[-1]
            # get files - these are the same in all outputs and sorted will alphabetize
            # 'hits' ie the TF calls will be index 0
            # 'mutagenesis' ie the actual predictions will be index -1
            parm_files = sorted(os.listdir(f'{folder}/{chunk}'))
            # get the mutagenesis file for that folder and open as a dataframe
            mut_file = pd.read_csv(f'{folder}/{chunk}/{parm_files[-1]}', sep = '\t')
            feature_cols = ['A', 'C', 'G', 'T']
            # Loop through each possible base
            for base in feature_cols:
                # Find rows where the 'Ref' column matches the current base
                # and set the value in the column with that base's name to 0
                mut_file.loc[mut_file['Ref'] == base, base] = 0
            # generate the 'raw' tensors by converting df to tensor
            raw_tensor = torch.tensor(mut_file.filter(['A', 'C', 'G', 'T']).to_numpy()).T
            # make seqlet tensor
            seqlet_tensor = raw_tensor.sum(dim=0) / 3
            # make oneHot tensor
            oneHot_bool = raw_tensor == 0
            oneHot_tensor = 1 * oneHot_bool
            # make plotLogo tensor
            plotLogo_tensor = seqlet_tensor * oneHot_tensor
            # get the corresponding enhancer
            enhID = pos_id_dict.get(element)
            # update dictionaries
            raw_tensor_dict[enhID] = raw_tensor
            seqlet_tensor_dict[enhID] = seqlet_tensor
            plotLogo_tensor_dict[enhID] = plotLogo_tensor
            oneHot_tensor_dict[enhID] = oneHot_tensor
    return raw_tensor_dict, seqlet_tensor_dict, plotLogo_tensor_dict, oneHot_tensor_dict;

In [ ]:
# generate tangermeme seqlet tensors for PARM distal enhancer preds
# K562
k562_parm_raw_tensors, k562_parm_seqlet_tensors, k562_parm_plotLogo_tensors, k562_parm_oneHot_tensors = parm2tangermeme_tensors(k562_folders,
                                                                                                                                ann_dict)

In [ ]:
# generate tangermeme seqlet tensors for PARM distal enhancer preds
# HepG2
hepg2_parm_raw_tensors, hepg2_parm_seqlet_tensors, hepg2_parm_plotLogo_tensors, hepg2_parm_oneHot_tensors = parm2tangermeme_tensors(hepg2_folders,
                                                                                                                                    ann_dict)

In [8]:
def jarOfPickles (path2pickles):
    pickleJar = {}
    pickles = glob.glob(os.path.join(path2pickles, "*.pkl"))

    print(f"Found {len(pickles)} pickle files to aggregate.")

    for file_path in tqdm(pickles, desc="pickling pickles"):
        with open(file_path, 'rb') as f:
            # Load the dictionary from the current pickle file
            chunk_dict = pickle.load(f)
            # Update the main dictionary with the contents of the chunk dictionary
            pickleJar.update(chunk_dict)
    return pickleJar

In [9]:
# open all pickles
# k562
k562_pickleJar = jarOfPickles('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/parm_preds/parm_tf_calls/k562')

Found 1470 pickle files to aggregate.


pickling pickles: 100%|██████████| 1470/1470 [46:09<00:00,  1.88s/it]  


In [ ]:
# open filtered TF calls generated with /scripts/parm_tf_processing/process_parm_chunk.py
PICKLE_DIR = "/path/to/your/pickle_output"
FINAL_OUTPUT_PATH = "/path/to/your/final_tf_call_dict.pkl"

# --- Main Logic ---
final_dict = {}
pickle_files = glob.glob(os.path.join(PICKLE_DIR, "*.pkl"))

print(f"Found {len(pickle_files)} pickle files to aggregate.")

for file_path in tqdm(pickle_files, desc="Aggregating pickles"):
    with open(file_path, 'rb') as f:
        # Load the dictionary from the current pickle file
        chunk_dict = pickle.load(f)
        # Update the main dictionary with the contents of the chunk dictionary
        final_dict.update(chunk_dict)

print(f"Aggregation complete. Final dictionary contains {len(final_dict)} keys.")
print(f"Saving final dictionary to {FINAL_OUTPUT_PATH}")

with open(FINAL_OUTPUT_PATH, 'wb') as f:
    pickle.dump(final_dict, f)

print("Done.")